This too is ollama interesting

In [6]:
import json
import re
from typing import List, Dict, Any
import ollama   # ✅ Ollama client

# =========================
# Model config (Llama 3 via Ollama)
# =========================
OLLAMA_MODEL = "llama3:latest"  # or "llama3:instruct-q4_K_M" for lower VRAM

System_Prompt = """
You are an expert keyword extractor specialized in chemical engineering.
Your task is to analyze abstracts from AIChE conferences and extract
exactly 5 or 10 of the most important and specific keywords or phrases related to
chemical engineering from each abstract, along with the unique countries
of all authors.

The keywords should:
1. Reflect the core chemical engineering focus of the abstract.
2. Each keyword should be either one word or two words—no longer phrases allowed.
3. Be highly specific (e.g., 'heterogeneous catalysis' instead of 'catalysis').
4. Avoid generic or vague terms (e.g., 'study', 'analysis', 'process').
5. Be formatted as a JSON list of exactly 5–10 unique entries without explanation.
"""

FEW_SHOT = """
### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: Copper nanocube electrodes selectively reduce CO2 to C2+ products via...
Output JSON:
{"keywords": ["CO2 electroreduction", "copper nanocubes", "C2+ products", "electrocatalysis", "selectivity tuning"],
 "countries": ["Switzerland"]}
"""

User_Prompt_template = """
{few_shot}
Extract exactly 5 to 10 most important chemical engineering keywords and all author countries from this abstract.
Format as JSON:
{{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "countries": ["country1", "country2", ...]}}

Input:
Title/Topic: {title_or_topic}
Authors/Affiliations: {authors_block}
Abstract: {abstract_text}

Output JSON:
"""

GENERIC = {
    "catalysis", "analysis", "study", "experiment", "experimental",
    "characterization", "materials", "material", "chemical engineering",
    "technique", "techniques", "process", "processes", "method", "methods",
    "optimization", "performance", "investigation", "properties", "system",
    "approach", "results", "paper", "model", "models"
}
KEEP_IF_CONTAINS = ["heterogeneous catalysis", "homogeneous catalysis"]

STOPWORDS = set("""
a an the and or for with of on into to from via in at by using use over under between among
this that these those our their your its is are was were be being been have has had will would
we they it as than also may can could should other more less based derived new novel toward towards
""".split())

# ===== Helpers =====
def _apply_chat(system: str, user: str) -> list:
    return [
        {"role": "system", "content": system.strip()},
        {"role": "user", "content": user.strip()},
    ]

def _extract_json(text: str) -> Dict[str, Any]:
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return {}
    block = m.group(0).strip().strip("`")
    try:
        return json.loads(block)
    except Exception:
        block = re.sub(r"'", '"', block)
        block = re.sub(r",\s*]", "]", block)
        block = re.sub(r",\s*}", "}", block)
        try:
            return json.loads(block)
        except Exception:
            return {}

def _clean_list(xs) -> List[str]:
    out, seen = [], set()
    for x in xs or []:
        k = re.sub(r"\s+", " ", str(x)).strip().strip(",;")
        if not k:
            continue
        lo = k.lower()
        if lo not in seen:
            seen.add(lo)
            out.append(k)
    return out

def _filter_generic(term: str) -> bool:
    lo = term.lower()
    if any(s in lo for s in KEEP_IF_CONTAINS):
        return True
    return lo not in GENERIC and len(lo) >= 3

def _authors_block(item: Dict[str, Any]) -> str:
    if not isinstance(item, dict):
        return ""
    parts = []
    if isinstance(item.get("authors_structured"), list):
        for a in item["authors_structured"]:
            nm = (a.get("name") or "").strip()
            af = (a.get("affiliation") or "").strip()
            if nm or af:
                parts.append(f"{nm} ({af})")
    elif item.get("presenting_author"):
        parts.append(str(item["presenting_author"]))
    return "; ".join(parts)[:800]

# ===== Ollama chat call =====
def _ollama_chat_once(messages: list) -> str:
    resp = ollama.chat(
        model=OLLAMA_MODEL,
        messages=messages,
        options={"temperature": 0},
        keep_alive="10m"
    )
    return resp["message"]["content"]

# ===== Core extraction =====
def extract_keywords_and_countries(title: str, abstract_text: str, authors_txt: str) -> Dict[str, List[str]]:
    user = User_Prompt_template.format(
        few_shot=FEW_SHOT.strip(),
        title_or_topic=(title or "").strip(),
        authors_block=(authors_txt or "").strip(),
        abstract_text=(abstract_text or "").strip()
    )
    messages = _apply_chat(System_Prompt, user)
    out = _ollama_chat_once(messages)

    data = _extract_json(out) or {}
    keywords = _clean_list(data.get("keywords", []))
    keywords = [k for k in keywords if k and _filter_generic(k)]
    countries = _clean_list(data.get("countries", []))
    return {"keywords": keywords, "countries": countries}

# ===== File handling =====
def process_file(path: str) -> List[Dict[str, List[str]]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Input JSON must be a list of objects.")

    results = []
    for item in data:
        if isinstance(item, dict):
            title = item.get("title") or item.get("topic") or ""
            abstract_text = item.get("abstract") or ""
            authors_txt = _authors_block(item)
        else:
            title, abstract_text, authors_txt = "", str(item), ""
        results.append(extract_keywords_and_countries(title, abstract_text, authors_txt))
    return results

def results_keywords_only(results: List[Dict[str, List[str]]]) -> List[List[str]]:
    return [r["keywords"] for r in results]

# ===== Main =====
if __name__ == "__main__":
    filename = "aiche_sample.json"  # Input file with abstracts
    save_full_results_to = "full_results.json"
    save_keywords_only_to = None

    all_results = process_file(filename)

    print("FULL RESULTS:")
    print(json.dumps(all_results, ensure_ascii=False, indent=2))

    if save_full_results_to:
        with open(save_full_results_to, "w", encoding="utf-8") as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)

    kw_matrix = results_keywords_only(all_results)
    print("\nKEYWORDS ONLY (array of arrays):")
    print(json.dumps(kw_matrix, ensure_ascii=False, indent=2))

    if save_keywords_only_to:
        with open(save_keywords_only_to, "w", encoding="utf-8") as f:
            json.dump(kw_matrix, f, ensure_ascii=False, indent=2)


FULL RESULTS:
[
  {
    "keywords": [
      "microengineered",
      "biomaterials",
      "regenerative medicine",
      "drug delivery",
      "tissue engineering"
    ],
    "countries": [
      "United States"
    ]
  },
  {
    "keywords": [
      "ionic liquids",
      "biomass dissolution",
      "thermodynamic properties",
      "transport properties",
      "Fickian diffusion"
    ],
    "countries": [
      "United States"
    ]
  },
  {
    "keywords": [
      "SLIPS",
      "biofouling",
      "nanoporous coatings",
      "layer-by-layer assembly",
      "antimicrobial agents",
      "nanoemulsions",
      "water-in-oil",
      "polymer tubing",
      "catheters"
    ],
    "countries": [
      "United States"
    ]
  },
  {
    "keywords": [
      "biomass pyrolysis",
      "hydrothermal carbonization",
      "pyrolyzed hydrochar",
      "adsorbent technologies",
      "techno-economic assessment"
    ],
    "countries": [
      "United States"
    ]
  },
  {
    "keywords

This the the ollama that i used to extract code imo

In [8]:
import os
import json
import re
from typing import List, Dict, Any
import ollama   # ✅ Ollama client

# =========================
# Model config (Llama 3 via Ollama)
# =========================
OLLAMA_MODEL = "llama3:latest"  # or "llama3:instruct-q4_K_M" for lower VRAM

ABSTRACT_YEAR = 2023

System_Prompt = """
You are an expert keyword extractor specialized in chemical engineering.
Your task is to analyze abstracts from AIChE conferences and extract
exactly 5 or 10 of the most important and specific keywords or phrases related to
chemical engineering from each abstract, along with the unique countries
of all authors.

The keywords should:
1. Reflect the core chemical engineering focus of the abstract.
2. Each keyword should be either one word or two words—no longer phrases allowed.
3. Be highly specific (e.g., 'heterogeneous catalysis' instead of 'catalysis').
4. Avoid generic or vague terms (e.g., 'study', 'analysis', 'process').
5. Be formatted as a JSON list of exactly 5–10 unique entries without explanation.
"""

FEW_SHOT = """
### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: Copper nanocube electrodes selectively reduce CO2 to C2+ products via...
Output JSON:
{"keywords": ["CO2 electroreduction", "copper nanocubes", "C2+ products", "electrocatalysis", "selectivity tuning"],
 "countries": ["Switzerland"]}
"""

User_Prompt_template = """
{few_shot}
Extract exactly 5 to 10 most important chemical engineering keywords and all author countries from this abstract.
Format as JSON:
{{"keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5"],
  "countries": ["country1", "country2", ...]}}

Input:
Title/Topic: {title_or_topic}
Authors/Affiliations: {authors_block}
Abstract: {abstract_text}

Output JSON:
"""

GENERIC = {
    "catalysis", "analysis", "study", "experiment", "experimental",
    "characterization", "materials", "material", "chemical engineering",
    "technique", "techniques", "process", "processes", "method", "methods",
    "optimization", "performance", "investigation", "properties", "system",
    "approach", "results", "paper", "model", "models"
}
KEEP_IF_CONTAINS = ["heterogeneous catalysis", "homogeneous catalysis"]

STOPWORDS = set(""" 
a an the and or for with of on into to from via in at by using use over under between among
this that these those our their your its is are was were be being been have has had will would
we they it as than also may can could should other more less based derived new novel toward towards
""".split())

# ===== Helpers =====
def _apply_chat(system: str, user: str) -> list:
    return [
        {"role": "system", "content": system.strip()},
        {"role": "user", "content": user.strip()},
    ]

def _extract_json(text: str) -> Dict[str, Any]:
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return {}
    block = m.group(0).strip().strip("`")
    try:
        return json.loads(block)
    except Exception:
        block = re.sub(r"'", '"', block)
        block = re.sub(r",\s*]", "]", block)
        block = re.sub(r",\s*}", "}", block)
        try:
            return json.loads(block)
        except Exception:
            return {}

def _clean_list(xs) -> List[str]:
    out, seen = [], set()
    for x in xs or []:
        k = re.sub(r"\s+", " ", str(x)).strip().strip(",;")
        if not k:
            continue
        lo = k.lower()
        if lo not in seen:
            seen.add(lo)
            out.append(k)
    return out

def _filter_generic(term: str) -> bool:
    lo = term.lower()
    if any(s in lo for s in KEEP_IF_CONTAINS):
        return True
    return lo not in GENERIC and len(lo) >= 3

def _authors_block(item: Dict[str, Any]) -> str:
    if not isinstance(item, dict):
        return ""
    parts = []
    if isinstance(item.get("authors_structured"), list):
        for a in item["authors_structured"]:
            nm = (a.get("name") or "").strip()
            af = (a.get("affiliation") or "").strip()
            if nm or af:
                parts.append(f"{nm} ({af})")
    elif item.get("presenting_author"):
        parts.append(str(item["presenting_author"]))
    # keep length reasonable for prompt
    return "; ".join(parts)[:800]

# ===== Ollama chat call =====
def _ollama_chat_once(messages: list) -> str:
    resp = ollama.chat(
        model=OLLAMA_MODEL,
        messages=messages,
        options={"temperature": 0},
        keep_alive="10m"
    )
    return resp["message"]["content"]

# ===== Core extraction =====
def extract_keywords_and_countries(title: str, abstract_text: str, authors_txt: str) -> Dict[str, List[str]]:
    user = User_Prompt_template.format(
        few_shot=FEW_SHOT.strip(),
        title_or_topic=(title or "").strip(),
        authors_block=(authors_txt or "").strip(),
        abstract_text=(abstract_text or "").strip()
    )
    messages = _apply_chat(System_Prompt, user)
    out = _ollama_chat_once(messages)

    data = _extract_json(out) or {}
    keywords = _clean_list(data.get("keywords", []))
    keywords = [k for k in keywords if k and _filter_generic(k)]
    countries = _clean_list(data.get("countries", []))
    return {"keywords": keywords, "countries": countries}

# ===== Progress bar helper =====
def _print_progress(current: int, total: int, kw_count: int, bar_width: int = 30):
    pct = (current / total) if total else 0
    filled = int(pct * bar_width)
    bar = "[" + "#" * filled + "-" * (bar_width - filled) + "]"
    print(f"{bar} {current}/{total} ({pct*100:5.1f}%)  | keywords this item: {kw_count}", end="\r")
    if current == total:
        print()  # newline on completion

# ===== File processing (single file) =====
def process_file(path: str, year: int = None) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Input JSON must be a list of objects.")

    results = []
    total_items = len(data)
    # initialize progress bar (0 processed)
    _print_progress(0, total_items, 0)

    for idx, item in enumerate(data, start=1):
        if isinstance(item, dict):
            title = (item.get("title") or item.get("topic") or "").strip()
            abstract_text = (item.get("abstract") or "").strip()
            authors_txt = _authors_block(item)
        else:
            title, abstract_text, authors_txt = "", str(item), ""

        # Extraction uses existing function exactly as before
        extracted = extract_keywords_and_countries(title, abstract_text, authors_txt)

        kw_count = len(extracted.get("keywords", []))
        result_item = {
            "title": title,
            "authors": authors_txt,
            "keywords": extracted.get("keywords", []),
            "countries": extracted.get("countries", []),
            "Year": ABSTRACT_YEAR if year is not None else None
        }

        results.append(result_item)

        # update progress bar after each item (show current item's keyword count)
        _print_progress(idx, total_items, kw_count)

    return results

# ===== Main runner for multiple files =====
if __name__ == "__main__":
    # User configuration: list of input files and output folder
    ABSTRACT_YEAR = 2023
    INPUT_FILES = [
        "aiche_papers_3311.json",
        "aiche_papers_3312.json",
        "aiche_papers_3313.json",
        "aiche_papers_3314.json",
        "aiche_papers_3315.json",
        "aiche_papers_3316.json",
        "aiche_papers_3317.json",
        "aiche_papers_3318.json",
        "aiche_papers_3319.json",
        "aiche_papers_3321.json",
        "aiche_papers_3325.json",
        "aiche_papers_3330.json"
    ]
    OUTPUT_FOLDER = "extracted"
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    combined = []
    total_files = len(INPUT_FILES)

    for file_idx, input_path in enumerate(INPUT_FILES, start=1):
        if not os.path.isfile(input_path):
            print(f"[{file_idx}/{total_files}] Missing: {input_path}")
            continue

        print(f"[{file_idx}/{total_files}] Extracting: {input_path}")
        per_file_results = process_file(input_path, year=ABSTRACT_YEAR)
        combined.extend(per_file_results)

        base = os.path.splitext(os.path.basename(input_path))[0]
        out_path = os.path.join(OUTPUT_FOLDER, f"{base}_full_results.json")

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(per_file_results, f, ensure_ascii=False, indent=2)

        print(f"Saved: {out_path}")

    # Save combined file
    COMBINED_OUT = os.path.join(OUTPUT_FOLDER, "all_combined_full_results.json")
    with open(COMBINED_OUT, "w", encoding="utf-8") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)

    print(f"Saved combined: {COMBINED_OUT}")


[1/12] Extracting: aiche_papers_3311.json
[##############################] 118/118 (100.0%)  | keywords this item: 5
Saved: extracted\aiche_papers_3311_full_results.json
[2/12] Extracting: aiche_papers_3312.json
[##############################] 112/112 (100.0%)  | keywords this item: 5
Saved: extracted\aiche_papers_3312_full_results.json
[3/12] Extracting: aiche_papers_3313.json
[##############################] 130/130 (100.0%)  | keywords this item: 5
Saved: extracted\aiche_papers_3313_full_results.json
[4/12] Extracting: aiche_papers_3314.json
[##############################] 57/57 (100.0%)  | keywords this item: 5
Saved: extracted\aiche_papers_3314_full_results.json
[5/12] Extracting: aiche_papers_3315.json
[##############################] 30/30 (100.0%)  | keywords this item: 6
Saved: extracted\aiche_papers_3315_full_results.json
[6/12] Extracting: aiche_papers_3316.json
[##############################] 212/212 (100.0%)  | keywords this item: 5
Saved: extracted\aiche_papers_3316_fu

THis thing is also ollama but i thinK not used

In [ ]:
# openrouter_aiche_extractor_ollama.py
# minimal changes from original + progress logs, switched to Ollama

import os
import json
from time import time
from typing import List, Dict, Any

# optional: keep dotenv in case you still want env configuration
from dotenv import load_dotenv
load_dotenv()

# --- Ollama client import ---
try:
    import ollama
except Exception as e:
    raise RuntimeError("Missing 'ollama' Python package. Install with: pip install ollama") from e

# Model you pulled locally with ollama pull <model>
# Keep the original MODEL_ID variable name for minimal changes.
MODEL_ID = os.getenv("OLLAMA_MODEL", "llama3:latest")  # update to the exact model you pulled, if needed

ABSTRACT_YEAR = 2023
INPUT_FILES = [
    "aiche_papers_3288.json"
]

OUTPUT_FOLDER = "extracted"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

SYSTEM_PROMPT = """You are an expert keyword extractor specialized in chemical engineering.
Analyze AIChE abstracts and output JSON with 'keywords' and 'countries'."""

FEW_SHOT = """### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: ...
Output JSON:
{"keywords":["CO2 electroreduction","copper nanocubes","C2+ products","electrocatalysis","selectivity tuning"],
 "countries":["Switzerland"]}"""

USER_TEMPLATE = """{few_shot}

Extract exactly 5–10 key chemical engineering keywords and all author countries.
Return JSON:
{{"keywords":["k1","k2"], "countries":["c1"]}}

Input:
Title/Topic: {title}
Authors/Affiliations: {authors}
Abstract: {abstract}

Output JSON:
"""

def authors_block(item):
    parts = []
    for a in item.get("authors_structured", []) or []:
        nm = (a.get("name") or "").strip()
        af = (a.get("affiliation") or "").strip()
        if nm or af:
            parts.append(f"{nm} ({af})")
    if not parts and item.get("presenting_author"):
        parts.append(str(item["presenting_author"]))
    return "; ".join(parts)

def call_ollama(messages, max_tokens=256, temperature=0.0):
    """
    Calls local Ollama via the Python client.
    Handles both non-streaming and streaming responses.
    Returns the assistant content (string).
    """
    # The ollama.chat API accepts messages similar to OpenAI-style chat messages.
    try:
        # Try synchronous call first
        resp = ollama.chat(model=MODEL_ID, messages=messages, max_tokens=max_tokens, temperature=temperature)
    except TypeError:
        # Older/newer client signature differences: try without max_tokens/temperature kwargs
        resp = ollama.chat(model=MODEL_ID, messages=messages)
    except Exception as e:
        # As a fallback, try streaming and join chunks (some clients return an iterator when stream=True)
        try:
            stream = ollama.chat(model=MODEL_ID, messages=messages, stream=True, max_tokens=max_tokens, temperature=temperature)
            parts = []
            for chunk in stream:
                # chunk may have structure: {'message': {'content': '...'}} or similar
                if isinstance(chunk, dict):
                    # try common keys
                    c = None
                    if "message" in chunk and isinstance(chunk["message"], dict):
                        c = chunk["message"].get("content")
                    elif "content" in chunk:
                        c = chunk.get("content")
                    if c:
                        parts.append(c)
            return "".join(parts)
        except Exception as e2:
            raise RuntimeError(f"Ollama call failed: {e}\nFallback also failed: {e2}") from e2

    # If resp is a dict (non-stream), try to extract content similar to OpenAI-style response
    if isinstance(resp, dict):
        # attempt several common patterns
        if "choices" in resp and resp["choices"]:
            choice = resp["choices"][0]
            # choice may be {'message': {'content': '...'}}
            if isinstance(choice, dict):
                msg = choice.get("message") or {}
                if isinstance(msg, dict) and "content" in msg:
                    return msg["content"]
                # or choice may have 'content' directly
                if "content" in choice:
                    return choice["content"]
        # Some clients return {'id':..., 'object':..., 'message':{'content':...}}
        if "message" in resp and isinstance(resp["message"], dict) and "content" in resp["message"]:
            return resp["message"]["content"]
        # Or simple {'response': '...'}
        if "response" in resp:
            return resp["response"]
        # Otherwise, stringify
        return str(resp)

    # If resp is an iterator (stream) - join
    try:
        parts = []
        for chunk in resp:
            if isinstance(chunk, dict):
                # common chunk formats
                if "message" in chunk and isinstance(chunk["message"], dict) and "content" in chunk["message"]:
                    parts.append(chunk["message"]["content"])
                elif "content" in chunk:
                    parts.append(chunk["content"])
                else:
                    parts.append(json.dumps(chunk))
            else:
                parts.append(str(chunk))
        return "".join(parts)
    except TypeError:
        # not iterable - just return string form
        return str(resp)

def parse_json_loose(text):
    try:
        return json.loads(text)
    except:
        import re
        s = text.strip().strip("`")
        m = re.search(r"\{[\s\S]*\}", s)
        if m:
            s = m.group(0)
        s = s.replace("’","'").replace("“",'"').replace("”",'"')
        s = s.replace("'",'"')
        s = re.sub(r",\s*}", "}", s)
        s = re.sub(r",\s*]", "]", s)
        try:
            return json.loads(s)
        except:
            return {"keywords": [], "countries": [], "_raw": text}

def process_file(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    results = []
    for i, item in enumerate(data, start=1):
        title = (item.get("title") or item.get("topic") or "").strip()
        abstract = (item.get("abstract") or "").strip()
        authors = authors_block(item)

        user_prompt = USER_TEMPLATE.format(
            few_shot=FEW_SHOT, title=title, authors=authors, abstract=abstract
        )

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        # call Ollama and parse
        try:
            raw = call_ollama(messages)
        except Exception as e:
            print(f"  [{i}/{len(data)}] Ollama call failed for item. Error: {e}")
            parsed = {"keywords": [], "countries": [], "_raw_error": str(e)}
        else:
            parsed = parse_json_loose(raw)

        results.append({
            "title": title,
            "authors": authors,
            "keywords": parsed.get("keywords", []),
            "countries": parsed.get("countries", []),
            "Year": ABSTRACT_YEAR
        })

        # small progress log per item
        print(f"  [{i}/{len(data)}] Processed: {title[:60]}... -> keywords: {len(parsed.get('keywords', []))}, countries: {len(parsed.get('countries', []))}")

    return results

# --- Run ---
if __name__ == "__main__":
    combined = []
    total = len(INPUT_FILES)

    for idx, input_path in enumerate(INPUT_FILES, start=1):
        if not os.path.isfile(input_path):
            print(f"[{idx}/{total}] Missing: {input_path}")
            continue

        print(f"[{idx}/{total}] Extracting: {input_path}")
        t0 = time()
        all_results = process_file(input_path)
        t1 = time()

        combined.extend(all_results)
        base = os.path.splitext(os.path.basename(input_path))[0]
        OUT_PATH = os.path.join(OUTPUT_FOLDER, f"{base}_full_results_ollama.json")

        with open(OUT_PATH, "w", encoding="utf-8") as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)

        print(f"Saved: {OUT_PATH} (took {t1-t0:.1f}s)")

    COMBINED_OUT = "all_combined_full_results.json"
    with open(COMBINED_OUT, "w", encoding="utf-8") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)

    print(f"Saved combined: {COMBINED_OUT}")


[1/1] Extracting: aiche_papers_3288.json


This is OLlama code


In [1]:
# openrouter_aiche_extractor_ollama.py
# minimal changes from original + progress logs, switched to Ollama

import os
import json
from time import time
from typing import List, Dict, Any

# optional: keep dotenv in case you still want env configuration
from dotenv import load_dotenv
load_dotenv()

# --- Ollama client import ---
try:
    import ollama
except Exception as e:
    raise RuntimeError("Missing 'ollama' Python package. Install with: pip install ollama") from e

# Model you pulled locally with ollama pull <model>
# Keep the original MODEL_ID variable name for minimal changes.
MODEL_ID = os.getenv("OLLAMA_MODEL", "llama3")  # update to the exact model you pulled, if needed

ABSTRACT_YEAR = 2023
INPUT_FILES = [
    "aiche_papers_3288.json",
    "aiche_papers_3298.json",
    "aiche_papers_3299.json",
    "aiche_papers_3300.json",
    "aiche_papers_3302.json",
    "aiche_papers_3303.json",
    "aiche_papers_3305.json",
    "aiche_papers_3306.json",
    "aiche_papers_3307.json",
    "aiche_papers_3309.json",
    "aiche_papers_3311.json",
    "aiche_papers_3312.json",
    "aiche_papers_3313.json",
    "aiche_papers_3314.json",
    "aiche_papers_3315.json",
    "aiche_papers_3316.json",
    "aiche_papers_3317.json",
    "aiche_papers_3318.json",
    "aiche_papers_3319.json",
    "aiche_papers_3321.json",
    "aiche_papers_3325.json",
    "aiche_papers_3330.json"
]

OUTPUT_FOLDER = "extracted"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

SYSTEM_PROMPT = """You are an expert keyword extractor specialized in chemical engineering.
Analyze AIChE abstracts and output JSON with 'keywords' and 'countries'."""

FEW_SHOT = """### Example
Input:
Title/Topic: CO2 Electroreduction to Multicarbon Products on Copper Nanocubes
Authors/Affiliations: M. Garcia (ETH Zürich, Switzerland)
Abstract: ...
Output JSON:
{"keywords":["CO2 electroreduction","copper nanocubes","C2+ products","electrocatalysis","selectivity tuning"],
 "countries":["Switzerland"]}"""

USER_TEMPLATE = """{few_shot}

Extract exactly 5–10 key chemical engineering keywords and all author countries.
Return JSON:
{{"keywords":["k1","k2"], "countries":["c1"]}}

Input:
Title/Topic: {title}
Authors/Affiliations: {authors}
Abstract: {abstract}

Output JSON:
"""

def authors_block(item):
    parts = []
    for a in item.get("authors_structured", []) or []:
        nm = (a.get("name") or "").strip()
        af = (a.get("affiliation") or "").strip()
        if nm or af:
            parts.append(f"{nm} ({af})")
    if not parts and item.get("presenting_author"):
        parts.append(str(item["presenting_author"]))
    return "; ".join(parts)

def call_ollama(messages, max_tokens=256, temperature=0.0):
    """
    Calls local Ollama via the Python client.
    Handles both non-streaming and streaming responses.
    Returns the assistant content (string).
    """
    # The ollama.chat API accepts messages similar to OpenAI-style chat messages.
    try:
        # Try synchronous call first
        resp = ollama.chat(model=MODEL_ID, messages=messages, max_tokens=max_tokens, temperature=temperature)
    except TypeError:
        # Older/newer client signature differences: try without max_tokens/temperature kwargs
        resp = ollama.chat(model=MODEL_ID, messages=messages)
    except Exception as e:
        # As a fallback, try streaming and join chunks (some clients return an iterator when stream=True)
        try:
            stream = ollama.chat(model=MODEL_ID, messages=messages, stream=True, max_tokens=max_tokens, temperature=temperature)
            parts = []
            for chunk in stream:
                # chunk may have structure: {'message': {'content': '...'}} or similar
                if isinstance(chunk, dict):
                    # try common keys
                    c = None
                    if "message" in chunk and isinstance(chunk["message"], dict):
                        c = chunk["message"].get("content")
                    elif "content" in chunk:
                        c = chunk.get("content")
                    if c:
                        parts.append(c)
            return "".join(parts)
        except Exception as e2:
            raise RuntimeError(f"Ollama call failed: {e}\nFallback also failed: {e2}") from e2

    # If resp is a dict (non-stream), try to extract content similar to OpenAI-style response
    if isinstance(resp, dict):
        # attempt several common patterns
        if "choices" in resp and resp["choices"]:
            choice = resp["choices"][0]
            # choice may be {'message': {'content': '...'}}
            if isinstance(choice, dict):
                msg = choice.get("message") or {}
                if isinstance(msg, dict) and "content" in msg:
                    return msg["content"]
                # or choice may have 'content' directly
                if "content" in choice:
                    return choice["content"]
        # Some clients return {'id':..., 'object':..., 'message':{'content':...}}
        if "message" in resp and isinstance(resp["message"], dict) and "content" in resp["message"]:
            return resp["message"]["content"]
        # Or simple {'response': '...'}
        if "response" in resp:
            return resp["response"]
        # Otherwise, stringify
        return str(resp)

    # If resp is an iterator (stream) - join
    try:
        parts = []
        for chunk in resp:
            if isinstance(chunk, dict):
                # common chunk formats
                if "message" in chunk and isinstance(chunk["message"], dict) and "content" in chunk["message"]:
                    parts.append(chunk["message"]["content"])
                elif "content" in chunk:
                    parts.append(chunk["content"])
                else:
                    parts.append(json.dumps(chunk))
            else:
                parts.append(str(chunk))
        return "".join(parts)
    except TypeError:
        # not iterable - just return string form
        return str(resp)

def parse_json_loose(text):
    try:
        return json.loads(text)
    except:
        import re
        s = text.strip().strip("`")
        m = re.search(r"\{[\s\S]*\}", s)
        if m:
            s = m.group(0)
        s = s.replace("’","'").replace("“",'"').replace("”",'"')
        s = s.replace("'",'"')
        s = re.sub(r",\s*}", "}", s)
        s = re.sub(r",\s*]", "]", s)
        try:
            return json.loads(s)
        except:
            return {"keywords": [], "countries": [], "_raw": text}

def process_file(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    results = []
    for i, item in enumerate(data, start=1):
        title = (item.get("title") or item.get("topic") or "").strip()
        abstract = (item.get("abstract") or "").strip()
        authors = authors_block(item)

        user_prompt = USER_TEMPLATE.format(
            few_shot=FEW_SHOT, title=title, authors=authors, abstract=abstract
        )

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        # call Ollama and parse
        try:
            raw = call_ollama(messages)
        except Exception as e:
            print(f"  [{i}/{len(data)}] Ollama call failed for item. Error: {e}")
            parsed = {"keywords": [], "countries": [], "_raw_error": str(e)}
        else:
            parsed = parse_json_loose(raw)

        results.append({
            "title": title,
            "authors": authors,
            "keywords": parsed.get("keywords", []),
            "countries": parsed.get("countries", []),
            "Year": ABSTRACT_YEAR
        })

        # small progress log per item
        print(f"  [{i}/{len(data)}] Processed: {title[:60]}... -> keywords: {len(parsed.get('keywords', []))}, countries: {len(parsed.get('countries', []))}")

    return results

# --- Run ---
if __name__ == "__main__":
    combined = []
    total = len(INPUT_FILES)

    for idx, input_path in enumerate(INPUT_FILES, start=1):
        if not os.path.isfile(input_path):
            print(f"[{idx}/{total}] Missing: {input_path}")
            continue

        print(f"[{idx}/{total}] Extracting: {input_path}")
        all_results = process_file(input_path)

        combined.extend(all_results)
        base = os.path.splitext(os.path.basename(input_path))[0]
        OUT_PATH = os.path.join(OUTPUT_FOLDER, f"{base}_full_results_ollama.json")

        with open(OUT_PATH, "w", encoding="utf-8") as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)

        print(f"Saved: {OUT_PATH} (took {t1-t0:.1f}s)")

    COMBINED_OUT = "all_combined_full_results.json"
    with open(COMBINED_OUT, "w", encoding="utf-8") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)

    print(f"Saved combined: {COMBINED_OUT}")


[1/22] Extracting: aiche_papers_3288.json
  [1/65] Processed: 224a- Predicting the Vapor-Liquid Equilibrium Curves of CO2a... -> keywords: 0, countries: 0
  [2/65] Processed: 224b- Calculations of Solubilities and Diffusion Coefficient... -> keywords: 0, countries: 0
  [3/65] Processed: 224c- Thermodynamic and Transport Properties of Ionic Liquid... -> keywords: 0, countries: 0
  [4/65] Processed: 224d- Cocrystal Formation Using Fatty Acid Toward a Sustaina... -> keywords: 0, countries: 0
  [5/65] Processed: 224e- A Tutorial on the Bayesian Approach to Inverse Problem... -> keywords: 0, countries: 0
  [6/65] Processed: 224f- Density-Pressure-Temperature Measurements of Binary Mi... -> keywords: 0, countries: 0
  [7/65] Processed: 224h- Interactive Thermodynamics Lecture Modules with Matlab... -> keywords: 0, countries: 0
  [8/65] Processed: 84a- Machine Learning with Weighted-Soap to Efficiently Pred... -> keywords: 0, countries: 0
  [9/65] Processed: 84c- Confined Fluid Phase Behavior

KeyboardInterrupt: 